In [1]:
# ============================================================
# 00 — Data Preparation (Memory Efficient Version)
# ============================================================

import pyreadr
import pandas as pd
import os
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "dataset").is_dir():
            return path
    raise FileNotFoundError("Could not find dataset/ from current directory")


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / "dataset"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# ── Training Data ─────────────────────────────────────────────
print("Processing training data...")

print("  Loading FaultFree_Training...")
res = pyreadr.read_r(str(RAW_DIR / "TEP_FaultFree_Training.RData"))
df_free = res['fault_free_training']
df_free['label'] = 0
df_free.to_csv(PROCESSED_DIR / "tep_train.csv", index=False)
print(f"  FaultFree written: {df_free.shape}")
del df_free, res

print("  Loading Faulty_Training...")
res = pyreadr.read_r(str(RAW_DIR / "TEP_Faulty_Training.RData"))
df_fault = res['faulty_training']
df_fault['label'] = df_fault['faultNumber'].astype(int)
df_fault.to_csv(
    PROCESSED_DIR / "tep_train.csv",
    mode="a",
    header=False,
    index=False,
)
print(f"  Faulty written: {df_fault.shape}")
del df_fault, res

print("Training CSV saved")

# ── Testing Data ──────────────────────────────────────────────
print("\nProcessing testing data...")

print("  Loading FaultFree_Testing...")
res = pyreadr.read_r(str(RAW_DIR / "TEP_FaultFree_Testing.RData"))
key = list(res.keys())[0]
print(f"  Object name: {key}")
df_free = res[key]
df_free['label'] = 0
df_free.to_csv(PROCESSED_DIR / "tep_test.csv", index=False)
print(f"  FaultFree written: {df_free.shape}")
del df_free, res

print("  Loading Faulty_Testing...")
res = pyreadr.read_r(str(RAW_DIR / "TEP_Faulty_Testing.RData"))
key = list(res.keys())[0]
print(f"  Object name: {key}")
df_fault = res[key]
df_fault['label'] = df_fault['faultNumber'].astype(int)
df_fault.to_csv(
    PROCESSED_DIR / "tep_test.csv",
    mode="a",
    header=False,
    index=False,
)
print(f"  Faulty written: {df_fault.shape}")
del df_fault, res

print("Testing CSV saved")

# ── Verify ────────────────────────────────────────────────────
print("\nVerifying files...")
train_size = os.path.getsize(PROCESSED_DIR / "tep_train.csv") / 1e6
test_size = os.path.getsize(PROCESSED_DIR / "tep_test.csv") / 1e6
print(f"tep_train.csv: {train_size:.0f} MB")
print(f"tep_test.csv:  {test_size:.0f} MB")
print("Done!")


Processing training data...
  Loading FaultFree_Training...
  FaultFree written: (250000, 56)
  Loading Faulty_Training...
  Faulty written: (5000000, 56)
Training CSV saved

Processing testing data...
  Loading FaultFree_Testing...
  Object name: fault_free_testing
  FaultFree written: (480000, 56)
  Loading Faulty_Testing...
  Object name: faulty_testing
  Faulty written: (9600000, 56)
Testing CSV saved

Verifying files...
tep_train.csv: 2002 MB
tep_test.csv:  3844 MB
Done!
